# Julia notebook to compute the symbolic solution to the frictional geostrophic equations with a specified buoyancy field and surface wind stress.

twnh Nov '25

This notebook solves

\begin{align}
- f  v & = - \frac{\partial p}{\partial x} + \epsilon^2   \nu \frac{d^2 u}{d z^2}, \\
  f  u & = - \frac{\partial p}{\partial y} + \epsilon^2   \nu \frac{d^2 v}{d z^2} , 
\end{align}
for $u(z), v(z)$, given viscosity $\nu = \nu_0$, and boundary conditions
\begin{align}
\text{Surface~}z = 0:
\begin{cases}
\displaystyle \epsilon^2 \nu \partial_z u & = \tau^x \\
\displaystyle  \epsilon^2 \nu \partial_z v & = \tau^y 
\end{cases}
 \\
\text{Bottom~}z = -H:
\begin{cases}
u & = 0  \\
v & = 0 
\end{cases} ,
\end{align}
where $(\tau^x, \tau^y)$ is the surface wind stress.
The pressure $p$ is due to the surface pressure field $p_s(x,y)$ and buoyancy field $b(x,y,z)$:

\begin{align}
p(x,y,z) & = p_s(x,y) + \int_{z}^{0} b(x,y,z') \; dz' , \\
\implies p_b(x, y) & \equiv p(x, y, z=-H(x,y)) = p_s(x,y) + \int_{-H(x,y)}^{0} b(x,y,z') \; dz' .
\end{align}

This code derives the equation satisfied by the surface pressure field $p_s(x,y)$.
It then solves for the pressure field given a specific simple example.

In [ ]:
using SymPy
im = SymPy.im  # SymPy's imaginary unit
notebook_name = "ExampleSolution_v0.2"
using Infiltrator
using Gridap
using GridapGmsh
using GridapMakie
using Gridap.Geometry
using WriteVTK

### Define symbols:

In [ ]:
# Geometry symbolic parameters:
z, ξ   = symbols("z ξ",   real=true, negative=true) # Vertical coordinate and source location (both in [-H,0])
x, y   = symbols("x y",   real=true)                # Horizontal coordinates
H      = SymFunction("H", real=true, positive=true) # Domain depth H(x)
geometry_params = (H(x,y), z, ξ)

# Frictional thermal wind equation symbolic parameters:
f, ϵ   = symbols("f ϵ",   real=true, positive=true) # Coriolis parameter and Ekman number
ν₀, ϕ  = symbols("ν₀ ϕ",  real=true, positive=true) # Viscosity parameters
τs     = SymFunction("τs",    complex=true)         # Complex surface wind stress
psg    = SymFunction("psg",   complex=true)         # Surface pressure gradient (∂/∂x + i ∂/∂y) pₛ(x,y)
b      = SymFunction("b",     real=true)            # Buoyancy field b(x,y,z)

# Define symbolic viscosity here:
ν = ν₀                                              # Constant viscosity profile
uv_params = (f, ϵ, ν) ;

### Compute pressure fields for the specified buoyancy field $b$:

In [ ]:
# Define buoyancy gradient field:
bg     = SymFunction("bg",    complex=true)         # Buoyancy gradient

# Compute pressure field:
pbarog = SymPy.integrate(bg(x,y,ξ),(ξ,z,0))         # Baroclinic presure gradient 
pg     = psg(x,y) + pbarog                          # Total pressure gradient
pbotg  = pg.subs(z,-H(x,y))                         # Bottom pressure gradient
# χ      = - SymPy.integrate(z * b(x,y,z),(z,-H(x,y),0))    # Baroclinic potential energy (diagnostic)

### Function to solve the frictional geostrophic equation using a Green's function:

In [ ]:
function compute_Guv(uv_params, geometry_params)
    # Setup symbols and parameters:
    f, ϵ, ν = uv_params
    H, z, ξ = geometry_params
    uv      = SymFunction("uv")
    A       = symbols("A",real=true)                                            # Unknown coefficient in the Green's function solution

    #0. Define the ODE for d/dz(uv(z)) = uv(z):
    ode = Eq(-im * f * uv(z) + ϵ^2 * diff(diff(ν * uv(z),z),z), 0)
    
    # 1. Solve for $G_- (z)$ on $-H \le z \le \xi$:
    Gₘ = dsolve(ode, uv(z), ics = Dict(uv(z).subs(z,-H(x,y))=>0)).rhs
    @assert simplify(ode.lhs.subs(uv(z),Gₘ)) == 0                               # Check solution
    # Replace constant names because otherwise they can interfere with the constants from the next dsolve below.
    const_names = collect([string(s) for s in Gₘ.free_symbols if occursin(r"^C\d+", string(s))])
    Gₘ = Gₘ.subs(const_names[1],A)
    @assert simplify(Gₘ.subs(z,-H(x,y))) == 0                                   # Check bottom BC

    # 2. Solve for $G_+(z)$ on  $\xi \le z \le 0$:
    Gₚ = dsolve(ode, uv(z), ics = Dict(diff(uv(z),z).subs(z,0)=>0)).rhs
    @assert simplify(ode.lhs.subs(uv(z),Gₚ)) == 0                               # Check solution
    @assert simplify(diff(Gₚ,z).subs(z,0)) == 0                                 # Check surface BC

    # 3. Compute Wronskian $W(z)$:
    W = Gₘ * diff(Gₚ, z) - Gₚ * diff(Gₘ, z)

    # 4. Compute Green's function $G(z; \xi)$:
    Gm = Gₘ * Gₚ.subs(z,ξ) / (ϵ^2 *  ν.subs(z,ξ) * W.subs(z,ξ))
    Gp = Gₘ.subs(z,ξ) * Gₚ / (ϵ^2 * ν.subs(z,ξ) * W.subs(z,ξ))


    # Check continuity and jump condition:
    @assert Gm.subs(z,ξ) - Gp.subs(z,ξ) == 0
    @assert simplify(diff(Gm, z).subs(z,ξ) - diff(Gp, z).subs(z,ξ)) + 1/(ϵ^2 * ν.subs(z,ξ)) == 0

    # #5. Define piecewise Green's function:
    G = sympy.Piecewise((Gm, Le(z,ξ)), (Gp, Ge(z,ξ)))
    
    # Check boundary conditions:
    @assert simplify(diff(G,z).subs(z,0).subs(ξ,-H//2)) == 0
    @assert simplify(G.subs(z,-H(x,y)).subs(ξ,-H//2)) == 0

    # Final simplify (to cancel constants). Avoid simplify in general because it's not always reproducible.
    G = simplify(G)
    return G
end ;

### Compute the G's function:

In [ ]:
Guv_sym = compute_Guv(uv_params,geometry_params) 
Guv = Guv_sym.subs(f,ϕ^2 * ϵ^2 * ν₀)
Guv0 = Guv.subs(ξ,0) 
tmp = sympy.integrate(expand(Guv), (ξ, -H(x,y), 0)).args[1].args[1]
max_obj = sympy.Max(z, -H(x, y))
Guv_int_wrt_ξ = tmp.subs(max_obj, z)

### Symbolic computation of flow $(u(z),v(z)), U, V, \tau_b$

In [ ]:
# Compute flow field u(z), v(z):
𝔲1 = SymPy.integrate(expand(Guv * pg.subs(z,ξ)),(ξ,-H(x,y),0))
max_obj = sympy.Max(z, -H(x, y))
𝔲1 = 𝔲1.subs(max_obj,z)
𝔲2 = - Guv0 * τs(x,y)
𝔲 = 𝔲1 + 𝔲2

# Check that the final expression for 𝔲 satisfies the original differential equation:
tmp1 = -im * f * 𝔲1 + ϵ^2 * diff(diff(ν * 𝔲1,z),z)
tmp1 = simplify(tmp1.subs(ϕ,sqrt(f/ν₀)/ϵ))
tmp2 = -im * f * 𝔲2 + ϵ^2 * diff(diff(ν * 𝔲2,z),z)
tmp2 = simplify(tmp2.subs(ϕ,sqrt(f/ν₀)/ϵ))
@assert tmp1 + tmp2 == pg

# Compute depth-integrated flow:
𝔘 = SymPy.integrate(expand(𝔲),(z,-H(x,y),0))

# Compute bottom stress on fluid:
τb = - ϵ^2 * ν * diff(𝔲,z).subs(z,-H(x,y))

# Check expression for surface stress on fluid:
@assert simplify(diff(𝔲1,z).subs(z,0)) == 0     # Pressure-driven part of surface stress vanishes
tmp = ϵ^2 * ν * diff(𝔲,z).subs(z,0)
@assert simplify(tmp - τs(x,y)) == 0

### Check final results from LaTeX derivation:

In [ ]:
tmp_var = ϕ*sympy.sqrt(im)
# Check integral of surface pressure term:
tmp_integrand = expand(exp(-tmp_var*ξ)*(exp(tmp_var*ξ) - exp(tmp_var*H(x,y)))*(exp(tmp_var*(ξ + H(x,y))) - 1))
tmp = sympy.integrate(tmp_integrand,(ξ,-H(x,y),0))
Latex_tmp = - H(x,y) * (exp(2*tmp_var*H(x,y)) + 1) + (1/tmp_var)*(exp(2*tmp_var*H(x,y)) - 1)
@assert tmp == Latex_tmp
display("LaTeX integral of the surface pressure term matches.")

# Check A(x) function:
Latex_A_fn = (im/f)*(H(x,y) + (1/tmp_var)*((1 - exp(2*tmp_var*H(x,y))) / (1 + exp(2*tmp_var*H(x,y)) ) )).subs(ϕ,sqrt(f/ν₀)/ϵ)
XXX = symbols("XXX")                            # SymPy can't pull the constant psg factor outside the integral.
𝔘_tmp = expand(𝔘.subs(psg(x,y),XXX).doit())
psg_coeff = 𝔘_tmp.coeff(XXX).subs(ϕ,sqrt(f/ν₀)/ϵ)
testA = simplify(Latex_A_fn - psg_coeff)
@assert testA == 0
display("LaTeX A(x) function matches.")

# Check B(x) function:
tmp_integrand2 = expand(tmp_integrand * sympy.integrate(bg(x,y,ξ),(ξ,ξ,0)))
Latex_B_fn = (-im/(f*(1+exp(2*tmp_var*H(x,y))))) * ( sympy.integrate(tmp_integrand2,(ξ,-H(x,y),0)) + τs(x,y)*(exp(tmp_var*H(x,y)) - 1)^2)
Latex_B_fn = Latex_B_fn.subs(ϕ,sqrt(f/ν₀)/ϵ)
𝔘_tmp2 = (𝔘_tmp - psg_coeff * XXX).doit()
𝔘_tmp2 = 𝔘_tmp2.subs(ϕ,sqrt(f/ν₀)/ϵ)
testB = 𝔘_tmp2 - Latex_B_fn
YYY = symbols("YYY")                            # SymPy can't work on the buried bg integral, so substitute for it
testB = testB.subs(bg(x,y,ξ),YYY)
testB = expand(testB).doit()
@assert testB == 0
display("LaTeX B(x) function matches.")

### Define parameter values:

In [ ]:
# Define the parameter values
f_val  = 1
ϵ_val  = 0.95
ν₀_val = 0.06
# ν₀_val = 0.01
println()
display("Non-dimensional Ekman-layer depth:")
Ekman_depth = sqrt(2*ν₀_val/f_val)
display(Ekman_depth)

# Compute compound parameter:
ϕ_val = sqrt(f_val / ν₀_val) / ϵ_val

# Define domain depth:
H_val(x,y)    = 1.0 - x^2 - y^2
# H_val(x,y)    = 1.0

# Define wind stress:
# τs_val(x,y) = 0.1 + 0.0im
# τs_val(x,y) = 0.0 + 0.0im
function τs_val(x, y)
    r = sqrt(x^2 + y^2)
    V = r^2           # Define this function for your use-case
    return -V * y / r + im* V * x / r
end

# Define buoyancy field: HARD CODED TO VANISH HERE (e.g., see calculation of velocity field below for plotting)
bg_val(x,y,z) = 0.0
b_val( x,y,z) = 0.0

# Define dictionaries for substitutions:
param_values = Dict(f=>f_val, ϵ=>ϵ_val, ν₀=>ν₀_val, ϕ=>ϕ_val)
fn_values    = Dict(τs(x,y)=>τs_val(x,y), H(x, y)=>H_val(x,y), bg(x,y,ξ)=>bg_val(x,y,ξ), b(x, y, z)=>b_val(x, y, z))

display("This case parameter values:")
display(param_values)

display("This case function values:")
display(fn_values)

### Solve for surface pressure using Gridap

In [ ]:
println("Solving for the surface pressure using Gridap...")

# 1. Define the mesh
model = GmshDiscreteModel("unit_circle_v0.2.msh")
Ω = Triangulation(model)
dΩ = Measure(Ω, 2)

# 2. Define the finite element space (piecewise linear,Dirichlet zero BC)
order = 2
reffe = ReferenceFE(lagrangian, Float64, order)

V = TestFESpace(
    model, reffe; conformity=:H1, dirichlet_tags="boundary"
)
U = TrialFESpace(V)

# 3. Define the coefficients of the elliptic equation:
A_fn_tmp = lambdify(Latex_A_fn.subs(fn_values).subs(param_values).doit(), [x, y])
A_fn(xx) = (typeof(xx[1]) <: Real ? A_fn_tmp(xx[1], xx[2]) : 1.0)     # Might get called with non-Float argument
B_fn_tmp = lambdify(Latex_B_fn.subs(fn_values).subs(param_values).doit(), [x, y])
B_fn(xx) = (typeof(xx[1]) <: Real ? B_fn_tmp(xx[1], xx[2]) : 0.0)     # Might get called with non-Float argument

# 4. Define weak form (variational formulation)
a(u,v) = ∫( real( (∇(v) ⋅ VectorValue( 1.0, -1im)) * (A_fn * (∇(u) ⋅ VectorValue(1.0, 1im))) ) ) * dΩ
l(v)   = ∫( real( (∇(v) ⋅ VectorValue(-1.0,  1im)) *  B_fn ) ) * dΩ

# 5. Assemble and solve
op = AffineFEOperator(a, l, U, V)
@time psurf = Gridap.solve(op)  # This is your numerical solution as a Gridap FEFunction

# 6. Visualization with Paraview
writevtk(Ω,notebook_name * "_psurf_solution",cellfields=["psurf"=>psurf])

### Solve for velocity field using the Green's function:

In [ ]:
println("Computing velocity field from surface pressure, Green's function, and known parameter and fields:")

function ps_val(xx, yy)
	gp = evaluate(psurf, Point(xx, yy))
	return gp
end

function psg_val(xx, yy)
	tmp = evaluate(∇(psurf), Point(xx, yy))
	return tmp[1] + 1im*tmp[2] 
end

# Make Julia functions from the symbolic expressions to accelerate for loop:
Guv0_fn          = lambdify(Guv0.subs(         param_values).subs(fn_values), [x, y, z])
Guv_int_wrt_ξ_fn = lambdify(Guv_int_wrt_ξ.subs(param_values).subs(fn_values), [x, y, z])
UV_fn(xx,yy)     = A_fn_tmp(xx,yy)*psg_val(xx,yy) + B_fn_tmp(xx,yy)
# τb_num = τb.subs(fn_values).subs(param_values).doit()
# UV_fn            = lambdify(𝔘_num, [x, y])
# τb_fn            = lambdify(τb_num, [x, y])
# div_𝔘_fn         = lambdify(div_𝔘, [x, y])
# curl𝔘_fn         = lambdify(curl𝔘, [x, y])

# Nx, Ny, Nz = 128, 128, 64
Nx, Ny, Nz = 64, 64, 32
xs     = range(-1, 1, Nx)
ys     = range(-1, 1, Ny)
zs     = range(-1, 0, Nz)
us     = zeros(Nx, Ny, Nz)
vs     = zeros(Nx, Ny, Nz)
ps     = zeros(Nx, Ny, Nz)
Us     = zeros(Nx, Ny)
Vs     = zeros(Nx, Ny)
τxs    = zeros(Nx, Ny)
τys    = zeros(Nx, Ny)
τbxs   = zeros(Nx, Ny)
τbys   = zeros(Nx, Ny)
# χs     = zeros(Nx, Ny)
pss    = zeros(Nx, Ny)
div_𝔘s = zeros(Nx, Ny)
curl𝔘s = zeros(Nx, Ny)
A_fns_Re  = zeros(Nx, Ny)
A_fns_Im  = zeros(Nx, Ny)
B_fns_Re  = zeros(Nx, Ny)
B_fns_Im  = zeros(Nx, Ny)
∂x_m_i∂y_B_Re = zeros(Nx, Ny)
∂x_m_i∂y_B_Im = zeros(Nx, Ny)

testB = Latex_B_fn.subs(fn_values).subs(param_values).doit()
∂x_m_i∂y_testB = diff(testB,x) - im * diff(testB,y)
∂x_m_i∂y_testB_tmp = lambdify(∂x_m_i∂y_testB, [x, y])

@time begin
for (ix, xx) in enumerate(xs)
	for (iy, yy) in enumerate(ys)
		this_H = H_val(xx, yy)
		if this_H >= 0
			this_pss = ps_val( xx, yy)      # Interpolate or evaluate ps
			this_psg = psg_val(xx, yy)      # Interpolate or evaluate psg
			for (iz, zz) in enumerate(zs)
				if zz <= 0 && zz >= -this_H
					# Pressure driven part
					Iuv = Guv_int_wrt_ξ_fn(xx, yy, zz) * this_psg

					# Surface stress driven part
					S = Guv0_fn(xx, yy, zz) * τs_val(xx, yy)
					us[ix, iy, iz] = real(Iuv - S)
					vs[ix, iy, iz] = imag(Iuv - S)
				end
			end
			# Compute 2D fields here:
			# @infiltrate()
			UV             = UV_fn(xx,yy)
			Us[    ix, iy] = float(real(UV))
			Vs[    ix, iy] = float(imag(UV))
			τxs[   ix, iy] = float(real(τs_val(xx,yy)))
			τys[   ix, iy] = float(imag(τs_val(xx,yy)))
			# τb_val         = τb_fn(xx,yy)
			# τbxs[  ix, iy] = float(real(τb_val))
			# τbys[  ix, iy] = float(imag(τb_val))
			pss[   ix, iy] = this_pss
			# div_𝔘s[ix, iy] = div_𝔘_fn(xx,yy)
			# curl𝔘s[ix, iy] = curl𝔘_fn(xx,yy)
			A_fns_Re[ix, iy] = real(A_fn_tmp(xx,yy))
			A_fns_Im[ix, iy] = imag(A_fn_tmp(xx,yy))
			B_fns_Re[ix, iy] = real(B_fn_tmp(xx,yy))
			B_fns_Im[ix, iy] = imag(B_fn_tmp(xx,yy))
			∂x_m_i∂y_B_Re[ix, iy] = real(∂x_m_i∂y_testB_tmp(xx,yy))
			∂x_m_i∂y_B_Im[ix, iy] = imag(∂x_m_i∂y_testB_tmp(xx,yy))
		end
	end
end
end

# Write out solution for display by Paraview:
vtk_grid(notebook_name * "_3D_solution", xs, ys, zs) do vtk
	vtk["u_speed"]  = us
	vtk["v_speed"]  = vs
	# vtk["pressure"] = ps
end

vtk_grid(notebook_name * "_2D_solution", xs, ys) do vtk
	vtk["U_speed"]      = Us
	vtk["V_speed"]      = Vs
	vtk["x_sfc_stress"] = τxs
	vtk["y_sfc_stress"] = τys
	# vtk["x_bot_stress"] = τbxs
	# vtk["y_bot_stress"] = τbys
	# vtk["chi"]          = χs
	vtk["sfc_p"]        = pss
	# vtk["div_UV"]       = div_𝔘s
	# vtk["curl_UV"]      = curl𝔘s
	vtk["A_fn_Re"]    	= A_fns_Re
	vtk["A_fn_Im"]    	= A_fns_Im
	vtk["B_fn_Re"]     	= B_fns_Re
	vtk["B_fn_Im"]     	= B_fns_Im
	vtk["divB_fn_Re"]   = ∂x_m_i∂y_B_Re
	vtk["divB_fn_Im"]   = ∂x_m_i∂y_B_Im
end